# Advanced RAG

**Goal:** improve vanilla dense RAG with a stronger retrieval stack.

Advanced system:

```text
question
 -> dense top-50
 -> BM25 top-50
 -> Reciprocal Rank Fusion
 -> cross-encoder reranking
 -> optional quality gate
 -> top-k context
 -> generator
```

In [1]:
%pip install -r ../requirements.txt


[notice] A new release of pip is available: 25.2 -> 26.1
[notice] To update, run: pip3.12 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

Project root: /Users/polinakorobeinikova/IU/NLP/case-study/case-study-rag-factual-consistency


In [3]:
import json
import time

import pandas as pd
from tqdm.auto import tqdm

from src.config import (
    PROCESSED_DIR, INDEX_DIR, PREDICTIONS_DIR, METRICS_DIR,
    GENERATOR_MODEL, EMBEDDING_MODEL, RERANKER_MODEL
)
from src.data_utils import load_gold_doc_ids
from src.generation import (
    load_generator, build_rag_prompt,
    generate_answer, answer_loss_and_perplexity, estimate_model_size_mb
)
from src.metrics import (
    evaluate_qa_predictions, recall_at_k, support_recall_at_k, numeric_summary
)
from src.retrieval import (
    DenseRetriever, BM25Retriever, HybridRerankRetriever,
    format_context, retrieved_doc_ids
)
from src.analysis_utils import Timer, save_json

pd.set_option("display.max_colwidth", 160)

In [4]:
questions_df = pd.read_parquet(PROCESSED_DIR / "questions.parquet")
corpus_df = pd.read_parquet(PROCESSED_DIR / "corpus.parquet")

EVAL_N = min(2000, len(questions_df))
TOP_K = 5

CALIBRATION_N = min(500, max(0, len(questions_df) - EVAL_N))

eval_df = questions_df.head(EVAL_N).copy()
calib_df = questions_df.iloc[EVAL_N:EVAL_N + CALIBRATION_N].copy()

print("eval:", eval_df.shape)
print("calibration:", calib_df.shape)

eval: (2000, 7)
calibration: (500, 7)


In [5]:
dense_retriever = DenseRetriever(
    index_path=INDEX_DIR / "dense_faiss.index",
    doc_ids_path=INDEX_DIR / "dense_doc_ids.json",
    corpus_df=corpus_df,
    model_name=EMBEDDING_MODEL,
)

bm25_retriever = BM25Retriever(
    bm25_path=INDEX_DIR / "bm25.pkl",
    corpus_df=corpus_df,
)

advanced_retriever = HybridRerankRetriever(
    dense=dense_retriever,
    bm25=bm25_retriever,
    reranker_name=RERANKER_MODEL,
    dense_k=50,
    bm25_k=50,
    rrf_k=60,
    rerank_k=20,
)

tokenizer, model, device = load_generator(GENERATOR_MODEL)
print("Device:", device)
print("Approx model size MB:", estimate_model_size_mb(model))

Device: mps
Approx model size MB: 1884.58544921875


## Optional quality gate calibration

The gate is intentionally simple and transparent:

- collect top reranker scores on a small calibration subset;
- identify scores from failed retrieval cases;
- use a conservative threshold;
- if the best reranker score is too low, answer "I don't know" instead of forcing a hallucinated answer.

In [6]:
calib_rows = []

for _, row in tqdm(calib_df.iterrows(), total=len(calib_df), desc="Calibrating gate"):
    gold_doc_ids = load_gold_doc_ids(row["gold_doc_ids"])
    chunks = advanced_retriever.retrieve(row["question"], top_k=TOP_K)
    ret_doc_ids = retrieved_doc_ids(chunks)
    best_score = chunks[0].get("rerank_score", float("-inf")) if chunks else float("-inf")

    calib_rows.append({
        "sample_id": row["sample_id"],
        "best_rerank_score": best_score,
        "hit_at_5": recall_at_k(ret_doc_ids, gold_doc_ids, 5),
        "support_recall_at_5": support_recall_at_k(ret_doc_ids, gold_doc_ids, 5),
    })

calib_scores_df = pd.DataFrame(calib_rows)
display(calib_scores_df.describe())
display(calib_scores_df.groupby("hit_at_5")["best_rerank_score"].describe())

Calibrating gate:   0%|          | 0/500 [00:00<?, ?it/s]

,best_rerank_score,hit_at_5,support_recall_at_5
count,500.000000,500.000000,500.000000
mean,5.555366,0.982000,0.783000
std,2.183860,0.133084,0.265616
min,-2.324294,0.000000,0.000000
25%,4.171751,1.000000,0.500000
50%,5.724521,1.000000,1.000000
75%,7.116224,1.000000,1.000000
max,10.525899,1.000000,1.000000


,count,mean,std,min,25%,50%,75%,max
hit_at_5,,,,,,,,
0.0,9.0,3.081699,3.568029,-2.324294,1.132685,4.446175,5.772076,7.115247
1.0,491.0,5.600708,2.129447,-1.859592,4.199264,5.737367,7.135624,10.525899


In [7]:
# Conservative threshold:
# If there are failed retrievals in calibration, use the 75th percentile of their top scores.
# Otherwise, disable the gate with a very low threshold.
failed_scores = calib_scores_df.loc[calib_scores_df["hit_at_5"] == 0, "best_rerank_score"]

if len(failed_scores) > 0:
    RERANK_GATE_THRESHOLD = float(failed_scores.quantile(0.75))
else:
    RERANK_GATE_THRESHOLD = float("-inf")

ENABLE_QUALITY_GATE = True

print("ENABLE_QUALITY_GATE:", ENABLE_QUALITY_GATE)
print("RERANK_GATE_THRESHOLD:", RERANK_GATE_THRESHOLD)

ENABLE_QUALITY_GATE: True
RERANK_GATE_THRESHOLD: 5.772075653076172


In [ ]:
rows = []

for _, row in tqdm(eval_df.iterrows(), total=len(eval_df), desc="Advanced RAG"):
    gold_doc_ids = load_gold_doc_ids(row["gold_doc_ids"])

    with Timer() as rt:
        chunks = advanced_retriever.retrieve(row["question"], top_k=TOP_K)
    retrieval_latency = rt.elapsed

    ret_doc_ids = retrieved_doc_ids(chunks)
    best_rerank_score = chunks[0].get("rerank_score", float("-inf")) if chunks else float("-inf")

    context = format_context(chunks)
    prompt = build_rag_prompt(row["question"], context)

    ppl_info = answer_loss_and_perplexity(
        tokenizer=tokenizer,
        model=model,
        prompt=prompt,
        answer=row["answer"],
    )

    gated = ENABLE_QUALITY_GATE and (best_rerank_score < RERANK_GATE_THRESHOLD)

    if gated:
        pred = "I don't know"
        gen_latency = 0.0
    else:
        pred, gen_latency = generate_answer(
            tokenizer=tokenizer,
            model=model,
            prompt=prompt,
            max_new_tokens=48,
            temperature=0.0,
        )

    rows.append({
        "system": "advanced_rag_hybrid_rerank_gate",
        "sample_id": row["sample_id"],
        "question": row["question"],
        "answer": row["answer"],
        "prediction": pred,
        "answer_loss": ppl_info["answer_loss"],
        "answer_ppl": ppl_info["answer_ppl"],
        "answer_n_tokens": ppl_info["answer_n_tokens"],
        "retrieved_doc_ids": json.dumps(ret_doc_ids, ensure_ascii=False),
        "gold_doc_ids": json.dumps(gold_doc_ids, ensure_ascii=False),
        "retrieved_context": context,
        "retrieval_hit_at_5": recall_at_k(ret_doc_ids, gold_doc_ids, 5),
        "support_recall_at_5": support_recall_at_k(ret_doc_ids, gold_doc_ids, 5),
        "best_rerank_score": best_rerank_score,
        "quality_gated": gated,
        "retrieval_latency_sec": retrieval_latency,
        "generation_latency_sec": gen_latency,
        "total_latency_sec": retrieval_latency + gen_latency,
    })

pred_df = pd.DataFrame(rows)
pred_df.to_csv(PREDICTIONS_DIR / "advanced_rag_predictions.csv", index=False)
calib_scores_df.to_csv(PREDICTIONS_DIR / "advanced_rag_gate_calibration.csv", index=False)

display(pred_df.head())

Advanced RAG:   0%|          | 0/2000 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


,system,sample_id,question,answer,prediction,answer_loss,answer_ppl,answer_n_tokens,retrieved_doc_ids,gold_doc_ids,retrieved_context,retrieval_hit_at_5,support_recall_at_5,best_rerank_score,quality_gated,retrieval_latency_sec,generation_latency_sec,total_latency_sec
0,advanced_rag_hybrid_rerank_gate,5a8b57f25542995d1e6f1371,Were Scott Derrickson and Ed Wood of the same nationality?,yes,I don't know,5.025447,152.238275,1,"[""5a8b57f25542995d1e6f1371::1::Scott Derrickson"", ""5a8b6d885542997f31a41d25::9::Ed Wood (film)"", ""5a8b57f25542995d1e6f1371::0::Ed Wood (film)"", ""5a84d2ea554...","[""5a8b57f25542995d1e6f1371::1::Scott Derrickson"", ""5a8b57f25542995d1e6f1371::4::Ed Wood""]","[1] Title: Scott Derrickson\nScott Derrickson (born July 16, 1966) is an American director, screenwriter and producer. He lives in Los Angeles, California....",1.0,1.0,3.394180,True,0.207974,0.000000,0.207974
1,advanced_rag_hybrid_rerank_gate,5a8c7595554299585d9e36b6,What government position was held by the woman who portrayed Corliss Archer in the film Kiss and Tell?,Chief of Protocol,I don't know,8.477502,4805.430296,3,"[""5a8c7595554299585d9e36b6::6::Kiss and Tell (1945 film)"", ""5ae1b1445542997283cd223d::7::A Kiss for Corliss"", ""5a8c7595554299585d9e36b6::5::A Kiss for Corli...","[""5a8c7595554299585d9e36b6::1::Shirley Temple"", ""5a8c7595554299585d9e36b6::6::Kiss and Tell (1945 film)""]","[1] Title: Kiss and Tell (1945 film)\nKiss and Tell is a 1945 American comedy film starring then 17-year-old Shirley Temple as Corliss Archer. In the film,...",1.0,0.5,3.197605,True,0.317686,0.000000,0.317686
2,advanced_rag_hybrid_rerank_gate,5a85ea095542994775f606a8,"What science fantasy young adult series, told in first person, has a set of companion books narrating the stories of enslaved worlds and alien species?",Animorphs,I don't know,0.402170,1.495066,3,"[""5a85ea095542994775f606a8::8::Animorphs"", ""5a8a547d55429970aeb70293::6::Talking to Dragons"", ""5a85ea095542994775f606a8::1::Victoria Hanley"", ""5a85ea0955429...","[""5a85ea095542994775f606a8::2::The Hork-Bajir Chronicles"", ""5a85ea095542994775f606a8::8::Animorphs""]","[1] Title: Animorphs\nAnimorphs is a science fantasy series of young adult books written by Katherine Applegate and her husband Michael Grant, writing toget...",1.0,1.0,5.451961,True,0.463181,0.000000,0.463181
3,advanced_rag_hybrid_rerank_gate,5adbf0a255429947ff17385a,Are the Laleli Mosque and Esma Sultan Mansion located in the same neighborhood?,no,I don't know,2.217420,9.183610,1,"[""5adbf0a255429947ff17385a::6::Esma Sultan Mansion"", ""5adbf0a255429947ff17385a::5::Laleli Mosque"", ""5adbf0a255429947ff17385a::7::Esma Sultan"", ""5adbf0a25542...","[""5adbf0a255429947ff17385a::5::Laleli Mosque"", ""5adbf0a255429947ff17385a::6::Esma Sultan Mansion""]","[1] Title: Esma Sultan Mansion\nThe Esma Sultan Mansion (Turkish: ""Esma Sultan Yalısı"" ), a historical yalı (English: waterside mansion ) located at Bosphor...",1.0,1.0,5.302992,True,0.267369,0.000000,0.267369
4,advanced_rag_hybrid_rerank_gate,5a8e3ea95542995a26add48d,"The director of the romantic comedy ""Big Stone Gap"" is based in what New York city?","Greenwich Village, New York City","Based on the information provided in the context, the director of the romantic comedy ""Big Stone Gap"" is based in New York City. The context states that the...",2.335597,10.335631,6,"[""5a8e3ea95542995a26add48d::9::Big Stone Gap (film)"", ""5a8e3ea95542995a26add48d::2::Nola (film)"", ""5a8e3ea95542995a26add48d::8::I Love NY (2015 film)"", ""5ad...","[""5a8e3ea95542995a26add48d::3::Adriana Trigiani"", ""5a8e3ea95542995a26add48d::9::Big Stone Gap (film)""]",[1] Title: Big Stone Gap (film)\nBig Stone Gap is a 2014 American drama romantic comedy film written and directed by Adriana Trigiani and produced by Donna ...,1.0,0.5,7.474613,False,0.341696,2.547382,2.889078


In [9]:
qa_metrics = evaluate_qa_predictions(pred_df.to_dict("records"))

metrics = {
    "system": "advanced_rag_hybrid_rerank_gate",
    "top_k": TOP_K,
}

metrics.update(qa_metrics)

metrics.update(numeric_summary(pred_df["answer_loss"], "answer_loss"))
metrics.update(numeric_summary(pred_df["answer_ppl"], "answer_ppl"))

metrics.update(numeric_summary(pred_df["retrieval_latency_sec"], "retrieval_latency_sec"))
metrics.update(numeric_summary(pred_df["generation_latency_sec"], "generation_latency_sec"))
metrics.update(numeric_summary(pred_df["total_latency_sec"], "total_latency_sec"))

metrics["retrieval_hit_at_5_mean"] = float(pred_df["retrieval_hit_at_5"].mean())
metrics["support_recall_at_5_mean"] = float(pred_df["support_recall_at_5"].mean())
metrics["quality_gated_rate"] = float(pred_df["quality_gated"].astype(float).mean())

# Reranker scores may contain -inf if no chunks were retrieved, so replace infinities for summary.
score_series = pd.to_numeric(pred_df["best_rerank_score"], errors="coerce").replace([float("inf"), float("-inf")], pd.NA).dropna()
metrics.update(numeric_summary(score_series, "best_rerank_score"))

metrics["generator_model"] = GENERATOR_MODEL
metrics["embedding_model"] = EMBEDDING_MODEL
metrics["reranker_model"] = RERANKER_MODEL
metrics["quality_gate_enabled"] = ENABLE_QUALITY_GATE
metrics["rerank_gate_threshold"] = RERANK_GATE_THRESHOLD

save_json(METRICS_DIR / "advanced_rag_metrics.json", metrics)
metrics

{'system': 'advanced_rag_hybrid_rerank_gate',
 'top_k': 5,
 'exact_match': 0.0595,
 'token_f1': 0.11182675100448695,
 'contains_answer': 0.173,
 'n': 2000,
 'answer_loss_mean': 2.6661574577651916,
 'answer_loss_median': 1.6785593032836914,
 'answer_loss_p90': 6.414990329742433,
 'answer_loss_p95': 8.36824378967285,
 'answer_loss_max': 16.151756286621094,
 'answer_ppl_mean': 22885.141168736478,
 'answer_ppl_median': 5.357831618168886,
 'answer_ppl_p90': 610.9365703197022,
 'answer_ppl_p95': 4308.259528633855,
 'answer_ppl_max': 10342335.65177612,
 'retrieval_latency_sec_mean': 0.7328609925279852,
 'retrieval_latency_sec_median': 0.5359872705002999,
 'retrieval_latency_sec_p90': 1.1293776339001853,
 'retrieval_latency_sec_p95': 1.4027777687495468,
 'retrieval_latency_sec_max': 15.945341333001124,
 'generation_latency_sec_mean': 1.567700740323513,
 'generation_latency_sec_median': 0.0,
 'generation_latency_sec_p90': 3.718789600300443,
 'generation_latency_sec_p95': 4.243551617001685,
 'ge

### Advanced RAG and quality gate analysis

The advanced RAG pipeline combines dense retrieval, BM25 lexical retrieval, reciprocal rank fusion, and cross-encoder reranking. The goal is to improve evidence completeness for HotpotQA, where many questions require multiple supporting paragraphs.

The first calibration attempt enabled a quality gate based on the best reranker score. However, this gate was too aggressive. It selected a threshold of 5.77 and rejected 51.7% of evaluation examples. Manual inspection showed that many rejected examples still had valid supporting evidence in the retrieved context. For example, some examples with SupportRecall@5 = 1.0 were still gated and answered as "I don't know".

This happened because the best reranker score did not cleanly separate retrieval failures from successful retrievals. Failed and successful examples had overlapping score distributions, so the threshold caused many false rejections. As a result, the gated version achieved only `EM = 0.0595` and token `F1 = 0.1118`, which is worse than vanilla RAG.

Therefore, the final advanced RAG system is reported without active gating. The main advanced result should be interpreted as the effect of hybrid retrieval and cross-encoder reranking, while the gate is treated as a negative ablation showing that naive score-based rejection can harm factual QA performance.

In [6]:
# Main advanced RAG result uses hybrid retrieval + reranking without active gating.
# The gate is analyzed separately because calibration showed that reranker score
# is not reliable enough as a rejection signal.
ENABLE_QUALITY_GATE = False

In [ ]:
rows = []

for _, row in tqdm(eval_df.iterrows(), total=len(eval_df), desc="Advanced RAG"):
    gold_doc_ids = load_gold_doc_ids(row["gold_doc_ids"])

    with Timer() as rt:
        chunks = advanced_retriever.retrieve(row["question"], top_k=TOP_K)
    retrieval_latency = rt.elapsed

    ret_doc_ids = retrieved_doc_ids(chunks)
    best_rerank_score = chunks[0].get("rerank_score", float("-inf")) if chunks else float("-inf")

    context = format_context(chunks)
    prompt = build_rag_prompt(row["question"], context)

    ppl_info = answer_loss_and_perplexity(
        tokenizer=tokenizer,
        model=model,
        prompt=prompt,
        answer=row["answer"],
    )

    gated = ENABLE_QUALITY_GATE and (best_rerank_score < RERANK_GATE_THRESHOLD)

    if gated:
        pred = "I don't know"
        gen_latency = 0.0
    else:
        pred, gen_latency = generate_answer(
            tokenizer=tokenizer,
            model=model,
            prompt=prompt,
            max_new_tokens=48,
            temperature=0.0,
        )

    rows.append({
        "system": "advanced_rag_hybrid_rerank",
        "sample_id": row["sample_id"],
        "question": row["question"],
        "answer": row["answer"],
        "prediction": pred,
        "answer_loss": ppl_info["answer_loss"],
        "answer_ppl": ppl_info["answer_ppl"],
        "answer_n_tokens": ppl_info["answer_n_tokens"],
        "retrieved_doc_ids": json.dumps(ret_doc_ids, ensure_ascii=False),
        "gold_doc_ids": json.dumps(gold_doc_ids, ensure_ascii=False),
        "retrieved_context": context,
        "retrieval_hit_at_5": recall_at_k(ret_doc_ids, gold_doc_ids, 5),
        "support_recall_at_5": support_recall_at_k(ret_doc_ids, gold_doc_ids, 5),
        "best_rerank_score": best_rerank_score,
        "quality_gated": gated,
        "retrieval_latency_sec": retrieval_latency,
        "generation_latency_sec": gen_latency,
        "total_latency_sec": retrieval_latency + gen_latency,
    })

pred_df = pd.DataFrame(rows)
pred_df.to_csv(PREDICTIONS_DIR / "advanced_rag_hybrid_rerank_predictions.csv", index=False)

In [8]:
display(pred_df.head())

,system,sample_id,question,answer,prediction,answer_loss,answer_ppl,answer_n_tokens,retrieved_doc_ids,gold_doc_ids,retrieved_context,retrieval_hit_at_5,support_recall_at_5,best_rerank_score,quality_gated,retrieval_latency_sec,generation_latency_sec,total_latency_sec
0,advanced_rag_hybrid_rerank,5a8b57f25542995d1e6f1371,Were Scott Derrickson and Ed Wood of the same nationality?,yes,No,5.025447,152.238275,1,"[""5a8b57f25542995d1e6f1371::1::Scott Derrickson"", ""5a8b6d885542997f31a41d25::9::Ed Wood (film)"", ""5a8b57f25542995d1e6f1371::0::Ed Wood (film)"", ""5a84d2ea554...","[""5a8b57f25542995d1e6f1371::1::Scott Derrickson"", ""5a8b57f25542995d1e6f1371::4::Ed Wood""]","[1] Title: Scott Derrickson\nScott Derrickson (born July 16, 1966) is an American director, screenwriter and producer. He lives in Los Angeles, California....",1.0,1.0,3.394180,False,0.545260,2.265392,2.810652
1,advanced_rag_hybrid_rerank,5a8c7595554299585d9e36b6,What government position was held by the woman who portrayed Corliss Archer in the film Kiss and Tell?,Chief of Protocol,The government position held by the woman who portrayed Corliss Archer in the film Kiss and Tell is that of a 17-year-old teenager. This can be inferred fro...,8.477502,4805.430296,3,"[""5a8c7595554299585d9e36b6::6::Kiss and Tell (1945 film)"", ""5ae1b1445542997283cd223d::7::A Kiss for Corliss"", ""5a8c7595554299585d9e36b6::5::A Kiss for Corli...","[""5a8c7595554299585d9e36b6::1::Shirley Temple"", ""5a8c7595554299585d9e36b6::6::Kiss and Tell (1945 film)""]","[1] Title: Kiss and Tell (1945 film)\nKiss and Tell is a 1945 American comedy film starring then 17-year-old Shirley Temple as Corliss Archer. In the film,...",1.0,0.5,3.197605,False,0.332091,2.264134,2.596225
2,advanced_rag_hybrid_rerank,5a85ea095542994775f606a8,"What science fantasy young adult series, told in first person, has a set of companion books narrating the stories of enslaved worlds and alien species?",Animorphs,Animorphs,0.402170,1.495066,3,"[""5a85ea095542994775f606a8::8::Animorphs"", ""5a8a547d55429970aeb70293::6::Talking to Dragons"", ""5a85ea095542994775f606a8::1::Victoria Hanley"", ""5a85ea0955429...","[""5a85ea095542994775f606a8::2::The Hork-Bajir Chronicles"", ""5a85ea095542994775f606a8::8::Animorphs""]","[1] Title: Animorphs\nAnimorphs is a science fantasy series of young adult books written by Katherine Applegate and her husband Michael Grant, writing toget...",1.0,1.0,5.451961,False,0.523789,2.608406,3.132195
3,advanced_rag_hybrid_rerank,5adbf0a255429947ff17385a,Are the Laleli Mosque and Esma Sultan Mansion located in the same neighborhood?,no,No.,2.217420,9.183610,1,"[""5adbf0a255429947ff17385a::6::Esma Sultan Mansion"", ""5adbf0a255429947ff17385a::5::Laleli Mosque"", ""5adbf0a255429947ff17385a::7::Esma Sultan"", ""5adbf0a25542...","[""5adbf0a255429947ff17385a::5::Laleli Mosque"", ""5adbf0a255429947ff17385a::6::Esma Sultan Mansion""]","[1] Title: Esma Sultan Mansion\nThe Esma Sultan Mansion (Turkish: ""Esma Sultan Yalısı"" ), a historical yalı (English: waterside mansion ) located at Bosphor...",1.0,1.0,5.302992,False,0.355440,2.113307,2.468747
4,advanced_rag_hybrid_rerank,5a8e3ea95542995a26add48d,"The director of the romantic comedy ""Big Stone Gap"" is based in what New York city?","Greenwich Village, New York City","Based on the information provided in the context, the director of the romantic comedy ""Big Stone Gap"" is based in New York City. The context states that the...",2.335597,10.335631,6,"[""5a8e3ea95542995a26add48d::9::Big Stone Gap (film)"", ""5a8e3ea95542995a26add48d::2::Nola (film)"", ""5a8e3ea95542995a26add48d::8::I Love NY (2015 film)"", ""5ad...","[""5a8e3ea95542995a26add48d::3::Adriana Trigiani"", ""5a8e3ea95542995a26add48d::9::Big Stone Gap (film)""]",[1] Title: Big Stone Gap (film)\nBig Stone Gap is a 2014 American drama romantic comedy film written and directed by Adriana Trigiani and produced by Donna ...,1.0,0.5,7.474613,False,0.373271,2.129850,2.503121


In [10]:
qa_metrics = evaluate_qa_predictions(pred_df.to_dict("records"))

metrics = {
    "system": "advanced_rag_hybrid_rerank",
    "top_k": TOP_K,
}

metrics.update(qa_metrics)

metrics.update(numeric_summary(pred_df["answer_loss"], "answer_loss"))
metrics.update(numeric_summary(pred_df["answer_ppl"], "answer_ppl"))

metrics.update(numeric_summary(pred_df["retrieval_latency_sec"], "retrieval_latency_sec"))
metrics.update(numeric_summary(pred_df["generation_latency_sec"], "generation_latency_sec"))
metrics.update(numeric_summary(pred_df["total_latency_sec"], "total_latency_sec"))

metrics["retrieval_hit_at_5_mean"] = float(pred_df["retrieval_hit_at_5"].mean())
metrics["support_recall_at_5_mean"] = float(pred_df["support_recall_at_5"].mean())
metrics["quality_gated_rate"] = float(pred_df["quality_gated"].astype(float).mean())

# Reranker scores may contain -inf if no chunks were retrieved, so replace infinities for summary.
score_series = pd.to_numeric(pred_df["best_rerank_score"], errors="coerce").replace([float("inf"), float("-inf")], pd.NA).dropna()
metrics.update(numeric_summary(score_series, "best_rerank_score"))

metrics["generator_model"] = GENERATOR_MODEL
metrics["embedding_model"] = EMBEDDING_MODEL
metrics["reranker_model"] = RERANKER_MODEL
metrics["quality_gate_enabled"] = ENABLE_QUALITY_GATE
metrics["rerank_gate_threshold"] = None

save_json(METRICS_DIR / "advanced_rag_hybrid_rerank_metrics.json", metrics)
metrics

{'system': 'advanced_rag_hybrid_rerank',
 'top_k': 5,
 'exact_match': 0.136,
 'token_f1': 0.23586640829852712,
 'contains_answer': 0.3145,
 'n': 2000,
 'answer_loss_mean': 2.6661574577651916,
 'answer_loss_median': 1.6785593032836914,
 'answer_loss_p90': 6.414990329742433,
 'answer_loss_p95': 8.36824378967285,
 'answer_loss_max': 16.151756286621094,
 'answer_ppl_mean': 22885.141168736478,
 'answer_ppl_median': 5.357831618168886,
 'answer_ppl_p90': 610.9365703197022,
 'answer_ppl_p95': 4308.259528633855,
 'answer_ppl_max': 10342335.65177612,
 'retrieval_latency_sec_mean': 0.7678346171944848,
 'retrieval_latency_sec_median': 0.5496069375003572,
 'retrieval_latency_sec_p90': 1.2640790793993801,
 'retrieval_latency_sec_p95': 1.8231625169997636,
 'retrieval_latency_sec_max': 18.102404333998493,
 'generation_latency_sec_mean': 2.756382621914507,
 'generation_latency_sec_median': 2.504816582999865,
 'generation_latency_sec_p90': 3.4721603287989637,
 'generation_latency_sec_p95': 3.98733836699

## Final advanced RAG result

The final advanced RAG system is reported without active quality gating. It uses hybrid retrieval and cross-encoder reranking, but it does not reject examples based on the reranker score.

Without active gating, advanced RAG achieves the best overall performance: Exact Match = 0.1360, token F1 = 0.2359, contains-answer = 0.3145, mean answer loss = 2.666, and median answer perplexity = 5.36 on 2,000 examples. This improves over vanilla dense RAG, which achieved Exact Match = 0.1060 and token F1 = 0.1849.

The cost is higher latency. Mean total latency increases to 3.52 seconds per query, compared with 2.77 seconds for vanilla RAG. This shows the main trade-off of advanced RAG: better factual consistency and evidence use, but higher retrieval and reranking overhead.